# DBSCAN
Accompanies the notes on **DBSCAN: Density-Based Clustering**.

## 1. Core, border and noise points: the hand example

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

P = np.array([[1, 1], [2, 1], [1, 2], [2.2, 2.0], [3.2, 2.0], [9, 9]])
db = DBSCAN(eps=1.5, min_samples=3).fit(P)
kind = np.array(["border"] * len(P), dtype=object)
kind[db.core_sample_indices_] = "core"
kind[db.labels_ == -1] = "noise"
for i, (k, lab) in enumerate(zip(kind, db.labels_)):
    print(f"p{i + 1}: {k:6s} cluster {lab}")

## 2. Choosing eps with the k-distance graph

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

X_raw, y_true = make_moons(n_samples=600, noise=0.08, random_state=42)
X_raw = np.vstack([X_raw, [[-1.2, 1.2], [2.3, -0.8], [0.5, 1.6], [2.4, 1.3]]])    # four stray points
X = StandardScaler().fit_transform(X_raw)

min_pts = 5
nn = NearestNeighbors(n_neighbors=min_pts).fit(X)          # each point counts as its own first neighbour,
distances, _ = nn.kneighbors(X)                            # as it does in DBSCAN's MinPts
k_dist = np.sort(distances[:, -1])
print("k-distance percentiles (50, 90, 95, 99, 100):", np.percentile(k_dist, [50, 90, 95, 99, 100]).round(3))
plt.figure(figsize=(7, 3.5))
plt.plot(k_dist); plt.axhline(0.2, color="crimson", ls="--", label="eps = 0.2")
plt.xlabel("points, sorted"); plt.ylabel(f"distance to {min_pts}th neighbour"); plt.legend()
plt.tight_layout(); plt.show()

## 3. K-means vs DBSCAN on the half-moons

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

km_labels = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X)
for eps in [0.1, 0.2, 0.3, 0.5]:
    labels = DBSCAN(eps=eps, min_samples=min_pts).fit_predict(X)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    print(f"eps={eps}: {n_clusters} clusters, {np.sum(labels == -1)} noise points, "
          f"agreement with the true moons (ARI, first 600 points) {adjusted_rand_score(y_true, labels[:600]):.3f}")
print("K-means, K=2: ARI", round(adjusted_rand_score(y_true, km_labels[:600]), 3))
db_labels = DBSCAN(eps=0.2, min_samples=min_pts).fit_predict(X)
print("labels of the four stray points:", db_labels[-4:])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(X[:, 0], X[:, 1], c=km_labels, s=8, cmap="viridis"); axes[0].set_title("K-means, K = 2")
axes[1].scatter(X[:, 0], X[:, 1], c=db_labels, s=8, cmap="viridis")
axes[1].scatter(X[db_labels == -1, 0], X[db_labels == -1, 1], s=40, facecolors="none", edgecolors="crimson", label="noise")
axes[1].set_title("DBSCAN, eps = 0.2, MinPts = 5"); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Clusters of different density: DBSCAN vs HDBSCAN

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import HDBSCAN

# two tight clusters close together, and one spread-out cluster
Xd, yd = make_blobs(n_samples=[200, 200, 200], centers=[[0, 0], [1.6, 0], [7, 0]],
                    cluster_std=[0.25, 0.25, 1.3], random_state=0)
for eps in [0.2, 0.5, 1.0]:
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(Xd)
    print(f"DBSCAN eps={eps}: sizes {np.bincount(labels[labels >= 0]).tolist()}, noise {np.sum(labels == -1)}, "
          f"ARI {adjusted_rand_score(yd, labels):.3f}")
labels = HDBSCAN(min_cluster_size=20).fit_predict(Xd)
print(f"HDBSCAN: sizes {np.bincount(labels[labels >= 0]).tolist()}, noise {np.sum(labels == -1)}, ARI {adjusted_rand_score(yd, labels):.3f}")

## 5. The bank customers

In [ ]:
import pandas as pd
bd = pd.read_csv("../../data/bd_train.csv")
cols = ["Average.Credit.Card.Transaction", "Balance.Transfer", "Term.Deposit", "Life.Insurance",
        "Medical.Insurance", "Average.A.C.Balance", "Personal.Loan", "Investment.in.Mutual.Fund",
        "Investment.Tax.Saving.Bond", "Home.Loan", "Online.Purchase.Amount", "Investment.in.Commudity",
        "Investment.in.Equity", "Investment.in.Derivative", "Portfolio.Balance"]
logx = np.log1p(bd[cols].clip(lower=0))
xs = (logx - logx.mean()) / logx.std(ddof=0)
k_dist = np.sort(NearestNeighbors(n_neighbors=30).fit(xs).kneighbors(xs)[0][:, -1])
print("30-distance percentiles (50, 90, 95, 99):", np.percentile(k_dist, [50, 90, 95, 99]).round(2))
for eps in [2.0, 3.0, 4.0]:
    labels = DBSCAN(eps=eps, min_samples=30).fit_predict(xs)
    print(f"eps={eps}: cluster sizes {np.bincount(labels[labels >= 0]).tolist()}, noise {np.sum(labels == -1)}")

## Try it yourself
1. eps = 1.0 in section 1.
2. eps = 0.25 on the moons.
3. `HDBSCAN(min_cluster_size=20)` on the moons.